# Colab: load a Drive checkpoint
Configured for your supplied model directory `/content/drive/MyDrive/MT/OPUS-mt-tw-fr-tuned` and a Twi-to-French route. Confirm that the saved model was trained for this direction; directory contents have not been inspected here. Use a complete model/tokenizer export. Commit/push notebook changes first or upload this notebook and the updated project to Colab. Colab is interactive compute, not 24/7 hosting. [Colab FAQ](https://research.google.com/colaboratory/faq.html) describes web-UI restrictions on free runtimes. Sharing is off by default; do not enable it for sensitive data.

In [ ]:
from pathlib import Path
import subprocess
import os
from urllib.parse import urlparse

REPO_URL = 'https://github.com/gyasifred/TW-FR-MT.git'  # Set your pushed repository.
BRANCH = ''
REPO_DIR = Path('/content/TW-FR-MT')
parsed = urlparse(REPO_URL)
if parsed.scheme != 'https' or parsed.hostname != 'github.com' or parsed.username or parsed.password or parsed.query or parsed.fragment:
    raise ValueError('Use a credential-free HTTPS GitHub repository URL.')
if not REPO_DIR.exists():
    command = ['git', 'clone']
    if BRANCH:
        command += ['--branch', BRANCH, '--single-branch']
    subprocess.run(command + ['--', REPO_URL, str(REPO_DIR)], check=True)
else:
    print('Using existing checkout/upload unchanged:', REPO_DIR)
if (REPO_DIR / '.git').exists():
    if BRANCH:
        actual = subprocess.check_output(['git', '-C', str(REPO_DIR), 'branch', '--show-current'], text=True).strip()
        if actual != BRANCH:
            raise RuntimeError(f'Existing branch {actual!r} differs from {BRANCH!r}; resolve manually.')
    subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], check=True)
if not (REPO_DIR / 'pyproject.toml').is_file():
    raise RuntimeError('Updated package missing: push first or upload the complete updated working tree.')
os.chdir(REPO_DIR)


In [ ]:
%pip install -e '.[train,app]'
from importlib.metadata import version
assert version('transformers') == '4.57.1', 'Use the project pin and restart the runtime if needed.'


## Mount authorized Drive storage
Open [the supplied folder](https://drive.google.com/drive/folders/13irIvPsqnryP_NJ5y6PneFKrQDJs5Qm6?usp=drive_link) using your authorized account. Add a shortcut to My Drive if necessary. Set `DRIVE_ROOT` to the actual mounted folder/shortcut path, not a name guessed from its ID. Mounting grants this notebook access to your Drive; no public download helper is used.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import json
DRIVE_ROOT = Path('/content/drive/MyDrive/MT')
MODEL_ROOT = DRIVE_ROOT
CHECKPOINT_RELATIVE = 'OPUS-mt-tw-fr-tuned'
REGISTRY_PATH = Path('/content/twfr-registries/tw-fr.json')
DEVICE = 'auto'
ENABLE_PUBLIC_SHARE = False  # True creates a public link; opt in deliberately.
LAUNCH_UI = False  # Set True where your Colab plan permits interactive Gradio use.
if not DRIVE_ROOT.is_dir():
    raise FileNotFoundError('Mount Drive and confirm the MT directory exists.')
checkpoint = (MODEL_ROOT / CHECKPOINT_RELATIVE).resolve()
if Path(CHECKPOINT_RELATIVE).is_absolute() or not checkpoint.is_relative_to(MODEL_ROOT.resolve()):
    raise ValueError('Checkpoint must be relative to and stay inside MODEL_ROOT.')
if not checkpoint.is_dir() or not (checkpoint / 'config.json').is_file():
    raise FileNotFoundError(f'Choose a complete saved model/tokenizer export: {checkpoint}')
safe_weights = any((checkpoint / name).is_file() for name in ('model.safetensors', 'model.safetensors.index.json'))
if not safe_weights:
    legacy_weights = any((checkpoint / name).is_file() for name in ('pytorch_model.bin', 'pytorch_model.bin.index.json'))
    if legacy_weights:
        raise ValueError('Legacy .bin weights found. Convert your trusted checkpoint using the instructions below, then select the converted directory and rerun this cell.')
    raise FileNotFoundError('No supported Hugging Face model weights found in the selected directory.')
registry = {'routes': [{'id': 'tw-fr', 'label': 'Twi to French', 'source_lang': 'tw', 'target_lang': 'fr', 'steps': [
    {'source_lang': 'tw', 'target_lang': 'fr', 'checkpoint': CHECKPOINT_RELATIVE,
     'source_tokenizer_lang': None, 'target_tokenizer_lang': None, 'source_prefix': ''}
]}]}
REGISTRY_PATH.parent.mkdir(parents=True, exist_ok=True)
payload = json.dumps(registry, indent=2) + '\n'
if REGISTRY_PATH.exists():
    if REGISTRY_PATH.read_text(encoding='utf-8') != payload:
        raise FileExistsError('Registry differs; choose a new REGISTRY_PATH instead of overwriting it.')
else:
    with REGISTRY_PATH.open('x', encoding='utf-8') as handle:
        handle.write(payload)
print(REGISTRY_PATH)
print(payload)


## Older checkpoints and other directions
If the selected folder has `pytorch_model.bin` instead of safetensors, run the following in a new code cell only for your own trusted model. It creates a separate export and refuses to overwrite an existing destination; the original remains unchanged. The original folder must also contain the saved tokenizer and config.
```python
from twfr_mt.export import export_checkpoint
export_checkpoint('/content/drive/MyDrive/MT/OPUS-mt-tw-fr-tuned',
                  '/content/drive/MyDrive/MT/OPUS-mt-tw-fr-tuned-safe',
                  allow_legacy_bin=True)
```
Then change `CHECKPOINT_RELATIVE` above to `OPUS-mt-tw-fr-tuned-safe` and rerun that cell. Tokenizer codes and prefixes must match the model's actual training configuration. For French-to-Twi or an English pivot, add the appropriate route and its actual trained checkpoint(s); the app does not infer language support from folder names.

In [ ]:
from twfr_mt.ui import create_app
app = create_app(registry_path=str(REGISTRY_PATH), model_root=str(MODEL_ROOT), device=DEVICE)
if LAUNCH_UI:
    app.launch(share=ENABLE_PUBLIC_SHARE, max_file_size='1mb', enable_monitoring=False, show_error=False, blocked_paths=[str(MODEL_ROOT.resolve())])
else:
    print('App constructed; UI launch and public sharing remain disabled.')


Stop the app with `app.close()` when done. Public sharing exposes the app to people with the link; keep confidential text and credentials out of inputs and outputs. Runtime termination removes `/content` files, not verified Drive exports. Loading checkpoints from Drive can be slow; see `docs/COLAB.md` for I/O caveats and `docs/DEPLOYMENT.md` for authenticated export to a real hosting environment.